# Rossmann TC1 Manual Baseline Notes

Author: Sean Kraemer

This notebook is an evidence artifact for the `tc1_from_scratch` human baseline.
It records dataset inspection and plain-language reasoning only.
The final bank-id mapping lives in `manual_selection_worksheet.md`, not in this notebook.


In [ ]:
import pandas as pd

train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")
store = pd.read_csv("../store.csv")

print("train shape", train.shape)
print("test shape", test.shape)
print("store shape", store.shape)
print("train columns", sorted(train.columns.tolist()))


## Store Metadata And Calendar Surface

First-pass reasoning:

- the daily sales table is incomplete without `store.csv`, so the store lookup should be joined before downstream feature work
- `Date` is central to the task and needs to be parsed before extracting simple calendar parts
- keep the calendar block simple rather than immediately expanding every derived promo/date interaction


In [ ]:
train_joined = train.merge(store, on="Store", how="left")
test_joined = test.merge(store, on="Store", how="left")

for frame in (train_joined, test_joined):
    frame["Date"] = pd.to_datetime(frame["Date"], format="%Y-%m-%d")

print("joined train missingness")
print(train_joined.isnull().mean().sort_values(ascending=False).head(10))
print("date range", train_joined["Date"].min(), train_joined["Date"].max())
print("sample calendar values")
print(train_joined["Date"].dt.year.head())
print(train_joined["Date"].dt.month.head())
print(train_joined["Date"].dt.day.head())


## Closure Handling, Leakage, And Sparse Metadata

Second-pass reasoning:

- the training data contains a large deterministic closed-store regime, so a single closure rule is defensible in a manual baseline
- `Customers` is train-only and should not survive into the final baseline
- `CompetitionDistance` and competitor start month/year are the clearest store metadata gaps to repair if competition timing is kept
- promo-specific timing fields exist, but I stopped before forcing the entire promo repair block into the baseline


In [ ]:
print("closed-store share in train", (train_joined["Open"] == 0).mean())
print("missing Open in test", test_joined["Open"].isnull().sum())
print("Customers in train only", "Customers" in train_joined.columns, "Customers" in test_joined.columns)

store_missing_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]
print(train_joined[store_missing_cols].isnull().mean().sort_values(ascending=False))
print(train_joined[["StoreType", "Assortment", "StateHoliday"]].head())


## Manual Takeaways From Inspection

- join `store.csv` into the daily train/test tables
- parse `Date` and keep the basic year/month/day calendar parts
- keep one simple closed-store rule instead of multiple overlapping closure branches
- repair `CompetitionDistance` and the competitor start month/year before using competition timing
- encode the low-cardinality store and holiday columns with one simple tree-friendly strategy
- drop `Customers` because it is not available in the competition test set
- stop before the full promo-specific repair block and raw-`Date` cleanup in this first-pass baseline
